<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/HIDROCARBUROS/PP_Produccion_Lotes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN Y EXTRACTOR (ejecutar una vez por sesión) ═══
!pip install pdfplumber pypdf requests certifi cryptography pandas ipywidgets google-api-python-client --quiet
import os, io, re, json, time, html, hashlib, calendar, unicodedata
from datetime import date
from urllib.parse import urljoin, unquote
import requests
import pandas as pd
import pdfplumber
import ipywidgets as w
from IPython.display import display
print("✅ Librerías listas")

# ── Extractor de la tabla "PRODUCCIÓN FISCALIZADA MENSUAL DE HIDROCARBUROS LÍQUIDOS" ──
# Lee la tabla por coordenadas (no como texto plano) para no mezclar columnas con gráficos o notas.
try:
    import pypdf
except ImportError:                                  # sin pypdf se recorren todas las páginas (más lento)
    pypdf = None

MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO",
         "SETIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]
ALIAS_MES = {"SEPTIEMBRE": "SETIEMBRE", "ENE": "ENERO", "FEB": "FEBRERO", "MAR": "MARZO", "ABR": "ABRIL",
             "MAY": "MAYO", "JUN": "JUNIO", "JUL": "JULIO", "AGO": "AGOSTO", "SET": "SETIEMBRE",
             "SEP": "SETIEMBRE", "OCT": "OCTUBRE", "NOV": "NOVIEMBRE", "DIC": "DICIEMBRE"}
COL_EMPRESA = ("OPERADOR", "CONTRATISTA", "COMPANIA")
NUM = re.compile(r"^-?\d[\d,.]*$")
LOTE_OK = re.compile(r"^(\d{1,3}(-?[A-Z]{1,2}(/[A-Z])?)?|1-AB|[IVXL]+(/[IVXL]+)?|Z-\d{1,2}[A-Z]?)(\(T{1,2}\))?$")
OPER_OK = re.compile(r"^[A-ZÁÉÍÓÚÑÜ&.\-/]{2,}$")
ZONAS = ["SELVA NORTE", "SELVA CENTRAL", "SELVA SUR", "NOROESTE", "NOR-OESTE", "ZOCALO"]


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c)).upper()


def mes_de(t):
    t = sin_tildes(t).strip(".")
    t = ALIAS_MES.get(t, t)
    return t if t in MESES else None


def lote_norm(t):
    t = re.sub(r"\(\s*(LGN|COND\.?)\s*\)", "", t.upper())          # "31-C ( LGN )" -> "31-C"
    t = re.sub(r"\s*\(\d\)\s*$", "", t.strip())                     # "192 (1)" -> "192"
    t = t.rstrip(".,").replace(" ", "")
    t = re.sub(r"^31-?([A-Z])/(?:31-?)?([A-Z])$", r"31-\1/\2", t)    # "31B/D", "31B/31D" -> "31-B/D"
    return {"1AB": "1-AB"}.get(t, t)


def producto_de_lote(t):
    t = t.upper()
    return "CONDENSADO" if "COND" in t else ("LGN" if "LGN" in t else None)


def a_num(t, miles_punto=False):
    if t in ("-", "–"):
        return 0.0
    if miles_punto:
        return float(t.replace(".", "").replace(",", "."))
    return float(t.replace(",", ""))


ZONA_FIJA = {"1-AB": "SELVA NORTE", "192": "SELVA NORTE", "8": "SELVA NORTE", "67": "SELVA NORTE",
             "95": "SELVA NORTE", "102": "SELVA NORTE", "64": "SELVA NORTE", "39": "SELVA NORTE",
             "131": "SELVA CENTRAL", "31-B": "SELVA CENTRAL", "31-C": "SELVA CENTRAL", "31-D": "SELVA CENTRAL",
             "31-E": "SELVA CENTRAL", "31-B/D": "SELVA CENTRAL", "LOB/EA/LO": "NOROESTE",
             "88": "SELVA SUR", "56": "SELVA SUR", "57": "SELVA SUR", "58": "SELVA SUR"}


# Mismo campo con distinto contrato a lo largo del tiempo (para series históricas y rankings)
LOTE_HISTORICO = {"1-AB": "1-AB/192", "192": "1-AB/192",      # 1-AB terminó el 29/08/2015; 192 desde el 30/08/2015
                  "Z-2B": "Z-2B/Z-69", "Z-69": "Z-2B/Z-69",     # Z-2B pasó a Z-69 (PetroPerú, 2023)
                  "31-B": "31-B/D", "31-D": "31-B/D"}           # Maple: Maquía y Agua Caliente por separado hasta 2000


# Nombres de operadores: el PDF corta los nombres largos en dos renglones y tiene erratas; se homologan.
# Se conserva el nombre leído en OPERADOR_ORIGINAL.
OPERADOR_NORM = [
    (r"^PLUSPETRO$|^PLUSPETROL PERU( CORP\.?)?$", "PLUSPETROL"),
    (r"^PPN( EN( LIQUIDACI[OÓ]N)?)?$", "PLUSPETROL NORTE (EN LIQUIDACIÓN)"),
    (r"^UPLAND( OIL( AND( GAS)?)?)?$", "UPLAND OIL AND GAS"),
    (r"^CORPORACI[OÓ]N DEL( SUR)?$", "CORPORACIÓN DEL SUR"),
    (r"^PETROLERA MONTERR+ICO$", "PETROLERA MONTERRICO"),
    (r"^UNIENE?R?G[IÍ]A( ABC)?$", "UNIENERGÍA ABC"),
    (r"^UNIPETRO( ABC)?$", "UNIPETRO ABC"),
    (r"^PETRO-?TECH$", "PETROTECH"),
    (r"^G\.? ?M\.? ?P\.?( S\.? ?A\.?)?$", "GMP"),
    (r"^PETROPER[UÚ]$", "PETROPERÚ"),
    (r"^VIGO( ENERGY)?$", "VIGO ENERGY"),
    (r"^PACIFIC( STRATUS)?$", "PACIFIC STRATUS"),
]

def norm_operador(nombre):
    """Homologa un nombre de operador; los consorcios 'A / B' se homologan por partes."""
    partes = []
    for parte in str(nombre or "").split("/"):
        x = re.sub(r"\s+", " ", parte).strip().upper()
        for patron, bueno in OPERADOR_NORM:
            if re.match(patron, x):
                x = bueno
                break
        if x:
            partes.append(x)
    return " / ".join(partes)


def lote_historico(lote):
    base = re.sub(r"\(T+\)$", "", lote)                        # contratos temporales: I(T), VI(T), Z-69(T)
    return LOTE_HISTORICO.get(base, base)


def zona_de(lote):
    base = re.sub(r"\(T+\)$", "", lote)
    if base in ZONA_FIJA:
        return ZONA_FIJA[base]
    if base.startswith("Z-"):
        return "ZOCALO"
    if re.fullmatch(r"[IVXL]+(/[IVXL]+)?", base):
        return "NOROESTE"
    return ""


def paginas_candidatas(path, n):
    """Índices de páginas con la tabla mensual o el resumen, ubicados con una lectura rápida (pypdf).
    Los PDF anuales tienen ~90 páginas con gráficos y mapas: pdfplumber solo analiza estas."""
    if pypdf is None:
        return list(range(n))
    try:
        textos = [sin_tildes(p.extract_text() or "") for p in pypdf.PdfReader(path).pages]
    except Exception:
        return list(range(n))
    cand = [i for i, t in enumerate(textos)
            if ("MENSUAL" in t and ("HIDROCARBUROS LIQUIDOS" in t or "FISCALIZADA" in t))
            or "RESUMEN DE PRODUCCION" in t or re.search(r"FISCALIZADA DE HIDROCARBUROS LIQUIDOS\s*-", t)]
    if not cand and sum(len(t) for t in textos) > 500:
        return list(range(n))                        # texto raro: por seguridad, todas
    return cand


def operadores_resumen(pdf, paginas=None):
    """{lote: operador} de las tablas resumen (anual 'RESUMEN DE PRODUCCIÓN…', mensual 4.1)."""
    out = {}
    patron = re.compile(r"^([A-ZÁÉÍÓÚÑÜ&.\- ]+?)\s*(?:\(\d\))?\s+(\S+)\s+[\d,.]+\s+[\d,.]+\s+[\d.,]+$")
    for ip in (paginas if paginas is not None else range(len(pdf.pages))):
        t = pdf.pages[ip].extract_text() or ""
        cab = sin_tildes(t[:200])
        if not (re.search(r"RESUMEN DE PRODUCCION", cab)
                or re.search(r"PRODUCCION FISCALIZADA DE HIDROCARBUROS LIQUIDOS - [A-Z]+ \d{4}", cab)):
            continue
        for linea in t.splitlines():
            m = patron.match(linea.strip())
            if not m:
                continue
            oper, lote = m.group(1).strip(), lote_norm(m.group(2))
            for z in ["SELVA NORTE", "SELVA CENTRAL", "SELVA SUR", "NOROESTE", "ZÓCALO", "ZOCALO"]:
                oper = re.sub(r"^" + z + r"\s+", "", oper)
            if oper and LOTE_OK.match(lote) and OPER_OK.match(oper.split()[0]) and "TOTAL" not in oper:
                out[lote] = oper
    return out


def info_reporte(pdf):
    txt = sin_tildes(" ".join((pdf.pages[i].extract_text() or "") for i in range(min(2, len(pdf.pages)))))
    m = re.search(r"ESTADISTICA\s+ANUAL\s+DE\s+HIDROCARBUROS\s+(\d{4})", txt)
    if m:
        return "ANUAL", int(m.group(1)), 12
    m = re.search(r"ESTADISTICA\s+MENSUAL\s+DE\s+HIDROCARBUROS\s+(" + "|".join(MESES + ["SEPTIEMBRE"]) + r")\s+(\d{4})", txt)
    if m:
        return "MENSUAL", int(m.group(2)), MESES.index(mes_de(m.group(1))) + 1
    return None, None, None


def lineas(words, tol=2.5):
    out = []
    for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if out and abs(out[-1][0] - w["top"]) <= tol:
            out[-1][1].append(w)
        else:
            out.append([w["top"], [w]])
    return [(top, sorted(ws, key=lambda w: w["x0"])) for top, ws in out]


def palabras(pg):
    """Palabras con coordenadas "de lectura". Algunos anuales (p. ej. 2008) dibujan las tablas horizontales con
    el texto girado 90°: se leen en el sentido correcto y se rotan las coordenadas para que filas y columnas
    queden como en una página normal."""
    chars = pg.chars
    girados = [c for c in chars if not c.get("upright", True)]
    if len(girados) <= len(chars) / 2:
        return pg.extract_words()
    antihorario = sum(1 for c in girados if c["matrix"][1] > 0) >= len(girados) / 2
    if antihorario:                                  # se lee de abajo hacia arriba
        ws = pg.extract_words(char_dir_rotated="btt", line_dir_rotated="ltr")
        conv = lambda w: {"x0": pg.height - w["bottom"], "x1": pg.height - w["top"], "top": w["x0"], "bottom": w["x1"]}
    else:                                            # se lee de arriba hacia abajo
        ws = pg.extract_words(char_dir_rotated="ttb", line_dir_rotated="rtl")
        conv = lambda w: {"x0": w["top"], "x1": w["bottom"], "top": pg.width - w["x1"], "bottom": pg.width - w["x0"]}
    return [{**w, **conv(w)} for w in ws]


def titulo_tabla(ws):
    """Título de la página si es la tabla MENSUAL de producción fiscalizada de líquidos; si no, None."""
    t = sin_tildes(" ".join(w["text"] for w in ws[:20]))
    # 2016 titula "PRODUCCIÓN DE HIDROCARBUROS LIQUIDOS MENSUAL 2016" (sin "FISCALIZADA")
    if "MENSUAL" not in t or "CAMPO" in t or not ("FISCALIZADA" in t or "HIDROCARBUROS LIQUIDOS" in t):
        return None
    if re.search(r"MENSUAL DE GAS|GAS NATURAL", t.split("LOTE")[0][:150]):
        return None
    if "HIDROCARBUROS LIQUIDOS" in t or re.search(r"FISCALIZADA MENSUAL\s*-?\s*\d{4}", t):   # "… MENSUAL - 2010"
        return t
    return None


def pagina_tabla(pg):
    """Columnas de la tabla mensual por lote, o None si la página no la tiene."""
    ws = sorted(palabras(pg), key=lambda w: (round(w["top"]), w["x0"]))
    titulo = titulo_tabla(ws)
    if not titulo:
        return None
    for top, lw in lineas(ws):
        textos = [sin_tildes(w["text"]) for w in lw]
        meses = [mes_de(t) for t in textos]
        if "LOTE" in textos and "ENERO" in meses and "DICIEMBRE" in meses:
            col = {}
            for w, t, m in zip(lw, textos, meses):
                if m:
                    col[m] = w
                elif t in ("TOTAL", "LOTE", "ZONA") or t in COL_EMPRESA:
                    col["OPERADOR" if t in COL_EMPRESA else t] = w
            anio = re.findall(r"(?<!\d)(19\d\d|20\d\d)(?!\d)", titulo)
            return ws, top, col, (int(anio[0]) if anio else None)
    return None


def extraer(path):
    filas, totales = [], []
    anio_tabla = None
    with pdfplumber.open(path) as pdf:
        tipo, anio, mes_rep = info_reporte(pdf)
        candidatas = paginas_candidatas(path, len(pdf.pages))
        for ip in candidatas:
            pg = pdf.pages[ip]
            r = pagina_tabla(pg)
            if not r:
                continue
            ws, top_hdr, col, anio_tit = r
            anio_tabla = anio_tabla or anio_tit
            # separador de miles de la página: "19.993" (antiguos) o "19,993"
            nums = [w["text"] for w in ws if re.fullmatch(r"\d{1,3}([.,]\d{3})+(\.\d+)?", w["text"])]
            miles_punto = sum(1 for t in nums if re.fullmatch(r"\d{1,3}(\.\d{3})+", t)) > len(nums) / 2
            centros = {m: (col[m]["x0"] + col[m]["x1"]) / 2 for m in MESES + ["TOTAL"] if m in col}
            lim_lote_der = (col["LOTE"]["x1"] + col["ENERO"]["x0"]) / 2
            lim_oper_der = col["LOTE"]["x0"] - 12
            x_oper = col["OPERADOR"]["x0"] if "OPERADOR" in col else 0
            lim_zona_der = x_oper - 4 if "ZONA" in col else -1
            paso = (centros["DICIEMBRE"] - centros["ENERO"]) / 11
            producto, bloque = None, []
            etiquetas_oper, etiquetas_zona = [], []

            def cerrar_bloque():
                # filas sin operador: toman la etiqueta de operador suelta más cercana del bloque
                for f in bloque:
                    if not f["OPERADOR"] and etiquetas_oper:
                        f["OPERADOR"] = min(etiquetas_oper, key=lambda e: abs(e[0] - f["_top"]))[1]
                    if "ZONA" in col and etiquetas_zona:
                        f["ZONA"] = min(etiquetas_zona, key=lambda e: abs(e[0] - f["_top"]))[1]
                filas.extend(bloque)

            for top, lw in lineas(ws):
                if top <= top_hdr + 2:
                    continue
                txt = sin_tildes(" ".join(w["text"] for w in lw)).strip()
                izq = [w for w in lw if (w["x0"] + w["x1"]) / 2 < lim_lote_der]
                nums = [w for w in lw if (w["x0"] + w["x1"]) / 2 > lim_lote_der and (NUM.match(w["text"]) or w["text"] in "-–")]
                # quita "Pág. 24" u otro n° de página pegado a la izquierda del rótulo del producto
                cab = re.sub(r"^((PAG\.?|\d+)\s+)+", "", txt)
                hay_valores = any((w["x0"] + w["x1"]) / 2 > lim_lote_der for w in nums)
                if not hay_valores and (re.match(r"^PETROLEO(\s*\(B[A-Z]*\))?$", cab)
                                        or re.match(r"^LIQUIDOS DE GAS NATURAL", cab)):
                    cerrar_bloque(); bloque, etiquetas_oper = [], []
                    producto = "PETROLEO" if cab.startswith("PETROLEO") else "LGN"
                    continue
                if producto is None:
                    continue
                vals = {}
                for w in nums:
                    xc = (w["x0"] + w["x1"]) / 2
                    m = min(centros, key=lambda k: abs(centros[k] - xc))
                    if abs(centros[m] - xc) <= paso * 0.6:
                        try:
                            vals[m] = a_num(w["text"], miles_punto)
                        except ValueError:
                            pass
                txt_izq = sin_tildes(" ".join(w["text"] for w in izq)).strip()
                es_bd = re.search(r"\(\s*B/D\s*\)", txt_izq)
                if re.search(r"SUB-?TOTAL|^TOTAL|TOTAL\s*$|PROMEDIO|\(BPD\)", txt_izq) or es_bd:
                    if vals and "PROMEDIO" not in txt_izq and "BPD" not in txt_izq and not es_bd:
                        totales.append({"PAGINA": ip + 1, "PRODUCTO": producto, "ETIQUETA": txt_izq, **vals})
                    continue
                lote_ws = [w for w in izq if w["x0"] >= lim_oper_der - 2 and not re.fullmatch(r"\(\d\)", w["text"])]
                oper_ws = [w for w in izq if w["x1"] < lim_oper_der and w["x0"] >= lim_zona_der
                           and OPER_OK.match(w["text"])]
                zona_ws = [w for w in izq if w["x1"] <= lim_zona_der + 2 and OPER_OK.match(w["text"])]
                zona_txt = sin_tildes(" ".join(w["text"] for w in zona_ws))
                if zona_txt and any(z in zona_txt for z in ZONAS):
                    etiquetas_zona.append((top, zona_txt.replace("NOR-OESTE", "NOROESTE")))
                lote_txt = " ".join(w["text"] for w in lote_ws)
                if lote_ws and vals and LOTE_OK.match(lote_norm(lote_txt)):
                    meses = {m: vals.get(m) for m in MESES}
                    bloque.append({"_top": top, "PAGINA": ip + 1,
                                   "PRODUCTO": producto_de_lote(lote_txt) or producto,
                                   "ZONA": "", "OPERADOR": " ".join(w["text"] for w in oper_ws).strip(),
                                   "LOTE_ORIGINAL": lote_txt, "LOTE": lote_norm(lote_txt),
                                   **meses, "TOTAL": vals.get("TOTAL")})
                elif oper_ws and not vals and not lote_ws:
                    etiquetas_oper.append((top, " ".join(w["text"] for w in oper_ws)))
            cerrar_bloque()
        if not tipo and filas:                     # portada sin el título esperado: año de la tabla
            tipo, anio, mes_rep = "ANUAL", anio_tabla, 12
        if not tipo or not anio:
            letras = sum(len(pg.extract_text() or "") for pg in pdf.pages[:15])
            return {"TIPO": None, "MOTIVO": "ESCANEADO (sin texto)" if letras < 200 else "FORMATO ANTIGUO (sin tabla reconocible)"}
        oper_res = operadores_resumen(pdf, candidatas)
    for f in filas:
        f["ZONA"] = zona_de(f["LOTE"]) or f["ZONA"]
        if f["OPERADOR"] == "" or len(f["OPERADOR"]) < 3:
            f["OPERADOR"] = oper_res.get(f["LOTE"], f["OPERADOR"])
    # imputación: si falta exactamente un mes (dentro del período) y el total lo permite
    hasta = 12 if tipo == "ANUAL" else mes_rep
    for f in filas:
        f.pop("_top", None)
        dentro = MESES[:hasta]
        vacios = [m for m in dentro if f[m] is None]
        suma = sum(f[m] or 0 for m in dentro)
        f["ESTADO"] = "OK"
        if f["TOTAL"] is not None and abs(suma - f["TOTAL"]) > 0.5:
            if len(vacios) == 1:
                f[vacios[0]] = f["TOTAL"] - suma
                f["ESTADO"] = f"IMPUTADO {vacios[0]}"
            else:
                f["ESTADO"] = "REVISAR"
        if f["TOTAL"] is None:                     # tabla sin columna TOTAL visible
            f["TOTAL"] = suma
        for m in dentro:
            if f[m] is None:
                f[m] = 0.0
    return {"TIPO": tipo, "ANIO": anio, "MES_REPORTE": mes_rep, "filas": filas, "totales": totales}


In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
_años = list(range(date.today().year, 1993, -1))
_ancho = {"description_width": "190px"}
sel_anio_ini = w.Dropdown(options=_años, value=2012, description="Año inicio", style=_ancho)
sel_anio_fin = w.Dropdown(options=_años, value=_años[0], description="Año fin", style=_ancho)
chk_mensual = w.Checkbox(value=True, description="Año en curso con las estadísticas mensuales")
chk_pdf = w.Checkbox(value=True, description="Guardar los PDF en Drive (DATA/PDF)")
display(w.VBox([
    w.HTML("<b>Período</b>. Cada <b>anual</b> trae los 12 meses por lote; para el año en curso "
           "se usa la última <b>mensual</b> publicada (trae el acumulado del año)."),
    w.HBox([sel_anio_ini, sel_anio_fin]), chk_mensual, chk_pdf]))


In [ ]:
# ═══ BLOQUE 3 · DESCARGA, EXTRACCIÓN, CONTROL Y DRIVE ═══
# Si se corta, vuelve a ejecutarlo: los PDF ya guardados en Drive no se vuelven a descargar.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA (CSV + subcarpeta PDF)
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
PAGINA_EST = ("https://www.perupetro.com.pe/wps/portal/corporativo/PerupetroSite/estadisticas/"
              "estad%C3%ADstica%20petrolera/!ut/p/z1/pdFND8IgDAbgn1TYEPW4xDlc_BiKbvZiOBkSnR6Mv19YvK41sVwged5QKCB0gL1"
              "_h6t_hUfvb_F8Rn3ZZqUyRsm6ajYzYZVb1C5u5FpBOwAxUoUAJPKH_JsfATbTVL48aTqfAJGfiymdT4DIu4bpP4F_3p8A_vL_BEB6PC0gR"
              "5DqweYTBqQZEqAyhrtirxmwVDSQOwboOOjn_ThUJ8IqruIDQCD6fQ!!/dz/d5/L2dBISEvZ0FBIS9nQSEh/")
BASE = "https://www.perupetro.com.pe"
DEPARTAMENTO_ZONA = {"SELVA NORTE": "LORETO"}   # respaldo si el lote no está en DATA/PP_LOTES_GEO.csv (mapa de lotes)
PAUSA = 2                                        # segundos entre descargas
ruta = "/content/Perupetro"; os.makedirs(ruta, exist_ok=True)
A_INI, A_FIN = sel_anio_ini.value, sel_anio_fin.value
if A_INI > A_FIN:
    raise ValueError("El año de inicio es posterior al año fin.")
VERIFICAR_SSL = True     # False solo como último recurso si la cadena no se puede completar (desactiva la verificación)
CAB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/124.0 Safari/537.36"}

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import logging
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
if "PDF" not in datos_drive:
    datos_drive["PDF"] = drive.files().create(body={"name": "PDF", "parents": [DRIVE_FOLDER_ID],
                                                    "mimeType": "application/vnd.google-apps.folder"},
                                              fields="id", supportsAllDrives=True).execute()["id"]
PDF_FOLDER_ID = datos_drive["PDF"]
pdf_drive = archivos_en(PDF_FOLDER_ID)

# Departamento real de cada lote (notebook PP_Mapa_Lotes.ipynb → DATA/PP_LOTES_GEO.csv); si no está, se usa la zona
DEPTO_LOTE = {}
if "PP_LOTES_GEO.csv" in datos_drive:
    try:
        bajar_de_drive(datos_drive["PP_LOTES_GEO.csv"], os.path.join(ruta, "PP_LOTES_GEO.csv"))
        _g = pd.read_csv(os.path.join(ruta, "PP_LOTES_GEO.csv"), encoding="utf-8-sig", dtype=str)
        DEPTO_LOTE = {k: v for k, v in zip(_g["LOTE"], _g["DEPARTAMENTO"]) if isinstance(v, str) and v}
        print(f"🗺️  Departamentos por lote tomados del mapa: {len(DEPTO_LOTE)} lotes")
    except Exception as e:
        print(f"⚠️ No se pudo leer PP_LOTES_GEO.csv ({e}); se usa la zona")

def depto_de(lote, zona):
    base = re.sub(r"\(T+\)$", "", lote)
    for k in (lote, base):
        if k in DEPTO_LOTE:
            return DEPTO_LOTE[k], "MAPA"
    d = DEPARTAMENTO_ZONA.get(zona, "")
    return d, ("ZONA" if d else "")
print(f"☁️  Drive conectado · PDF ya guardados: {len(pdf_drive)}")

# ── 3. CERTIFICADO HTTPS DE PERUPETRO ───────────────────────────────────────
# El servidor no envía su certificado intermedio (los navegadores lo completan solos; Python no).
# Se descarga desde la dirección que indica el propio certificado (AIA) y se verifica con la cadena completa.
import ssl, socket, certifi
from cryptography import x509
from cryptography.hazmat.primitives.serialization import Encoding
from cryptography.x509.oid import ExtensionOID, AuthorityInformationAccessOID

def cert_del_servidor(host):
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    with socket.create_connection((host, 443), timeout=30) as s, ctx.wrap_socket(s, server_hostname=host) as ss:
        return x509.load_der_x509_certificate(ss.getpeercert(binary_form=True))

def emisores(cert):
    try:
        aia = cert.extensions.get_extension_for_oid(ExtensionOID.AUTHORITY_INFORMATION_ACCESS).value
    except x509.ExtensionNotFound:
        return []
    return [d.access_location.value for d in aia if d.access_method == AuthorityInformationAccessOID.CA_ISSUERS]

def leer_cert(datos):
    try:
        return x509.load_der_x509_certificate(datos)
    except ValueError:
        return x509.load_pem_x509_certificate(datos)

def bundle_con_intermedios(host, destino):
    cert, pems = cert_del_servidor(host), []
    for _ in range(3):                            # sube por la cadena hasta 3 niveles
        urls = emisores(cert)
        if not urls or cert.issuer == cert.subject:
            break
        cert = leer_cert(requests.get(urls[0], timeout=30).content)
        pems.append(cert.public_bytes(Encoding.PEM).decode())
    with open(destino, "w") as f:
        f.write(open(certifi.where()).read() + "\n" + "\n".join(pems))
    return destino, len(pems)

def preparar_verificacion():
    try:
        requests.get(BASE, headers=CAB, timeout=30)
        return True                               # la cadena ya es válida
    except requests.exceptions.SSLError:
        pass
    try:
        ruta_ca, n = bundle_con_intermedios("www.perupetro.com.pe", os.path.join(ruta, "ca_perupetro.pem"))
        requests.get(BASE, headers=CAB, timeout=30, verify=ruta_ca)
        print(f"🔒 Certificado de PERUPETRO completado con {n} intermedio(s): conexión verificada")
        return ruta_ca
    except Exception as e:
        if VERIFICAR_SSL:
            print(f"⚠️ No se pudo completar el certificado de PERUPETRO ({type(e).__name__}: {e}).\n"
                  "   Opciones: sube los PDF a mano a DATA/PDF, o pon VERIFICAR_SSL = False arriba (último recurso).")
            return True
        print("⚠️ VERIFICAR_SSL = False: se descarga SIN verificar el certificado (solo PDF públicos)")
        return False

VERIFY = preparar_verificacion()

# ── 4. ENLACES DE LA PÁGINA "ESTADÍSTICA PETROLERA" ─────────────────────────
MES_ABR = {"ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
           "SET": 9, "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12}

def clasificar(href, texto):
    """(tipo, año, mes) a partir del nombre del archivo y el texto del enlace; None si no se sabe."""
    nombre = sin_tildes(unquote(href).replace("+", " "))
    if re.search(r"REGLAMENTO|DECRETO|RESOLUCION|\bLEY\b|D\.S|PARTICIPACION", nombre):
        return None, None, None                   # documentos legales enlazados en la misma página
    anio = re.findall(r"(?<!\d)(199\d|20[0-4]\d)(?!\d)", nombre)
    t = sin_tildes(html.unescape(re.sub(r"<[^>]+>", " ", texto))).strip()
    if "MENSUAL" in nombre:
        m = re.search(r"(ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SET|SEP|OCT|NOV|DIC)[A-Z]*\s+(?:DE\s+)?(\d{4})", nombre)
        if m:
            return "MENSUAL", int(m.group(2)), MES_ABR[m.group(1)]
        return "MENSUAL", int(anio[-1]) if anio else None, MES_ABR.get(t[:3])
    if "ANUAL" in nombre or re.fullmatch(r"(19|20)\d\d", t):
        return "ANUAL", int(anio[-1]) if anio else (int(t) if t.isdigit() else None), None
    if anio:                                      # en esta página, un PDF con año y sin "mensual" es el anual
        return "ANUAL", int(anio[-1]), None
    return None, None, None

try:
    pagina = requests.get(PAGINA_EST, headers=CAB, timeout=60, verify=VERIFY).text
except Exception as e:
    pagina = ""
    print(f"⚠️ No se pudo leer la página de PERUPETRO ({e}). Se usarán solo los PDF que ya estén en Drive/PDF.")
enlaces = {}
for href, texto in re.findall(r'<a[^>]+href="([^"]+)"[^>]*>(.*?)</a>', pagina, re.S | re.I):
    href = html.unescape(href)
    if "/wps/wcm/connect/" in href and (".pdf" in href.lower() or "AJPERES" in href.upper()):
        url = urljoin(BASE, href)
        enlaces.setdefault(url, clasificar(href, texto))
print(f"🔗 Enlaces a PDF en la página: {len(enlaces)}")

# ── 5. DESCARGA (o reutiliza lo guardado en Drive) Y EXTRACCIÓN ─────────────
NOMBRE_CATALOGO = "PP_CATALOGO_PDF.csv"
catalogo = pd.DataFrame(columns=["URL", "ARCHIVO"])
if NOMBRE_CATALOGO in control_drive:
    try:                                          # un catálogo vacío (corrida que falló antes) no debe frenar
        leido = pd.read_csv(io.BytesIO(drive.files().get_media(fileId=control_drive[NOMBRE_CATALOGO],
                            supportsAllDrives=True).execute()), dtype=str)
        if {"URL", "ARCHIVO"} <= set(leido.columns):
            catalogo = leido.dropna(subset=["URL", "ARCHIVO"])
    except pd.errors.EmptyDataError:
        pass
url_a_archivo = dict(zip(catalogo["URL"], catalogo["ARCHIVO"]))
año_hoy = date.today().year

def en_rango(tipo, anio):
    if tipo is None:
        return False                              # no es una estadística
    if anio is None:
        return True                               # se decide al leer el PDF
    if tipo == "MENSUAL":
        return chk_mensual.value and A_INI <= anio <= A_FIN
    return A_INI <= anio <= A_FIN

def descargar(url):
    for i in range(1, 4):
        try:
            r = requests.get(url, headers=CAB, timeout=180, verify=VERIFY)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                return r.content
            print(f"   ↻ HTTP {r.status_code} ({i}/3)")
        except Exception as e:
            print(f"   ↻ {type(e).__name__} ({i}/3)")
        time.sleep(3 * i)
    return None

resultados, control = [], []
pendientes = [(u, c) for u, c in enlaces.items() if en_rango(c[0], c[1])]
# PDF puestos a mano en Drive/PDF que no vienen de la página
manuales = [n for n in pdf_drive if n.lower().endswith(".pdf") and n not in url_a_archivo.values()]
print(f"📄 A procesar: {len(pendientes)} de la página + {len(manuales)} subidos a mano en Drive/PDF\n")

def procesar(local, origen):
    t0 = time.time()
    try:
        r = extraer(local)
    except Exception as e:
        return None, {"ARCHIVO": origen, "ESTADO": f"ERROR: {type(e).__name__}: {e}"}
    if not r:
        return None, {"ARCHIVO": origen, "ESTADO": "NO ES ESTADÍSTICA DE HIDROCARBUROS"}
    if r["TIPO"] is None:                         # PDF antiguo: escaneado o con otro diseño
        return None, {"ARCHIVO": origen, "ESTADO": r["MOTIVO"]}
    return r, {"ARCHIVO": origen, "TIPO": r["TIPO"], "AÑO": r["ANIO"], "MES_REPORTE": r["MES_REPORTE"],
               "SEGUNDOS": round(time.time() - t0, 1)}

trabajos = [("web", u) for u, _ in pendientes] + [("drive", n) for n in manuales]
for n, (fuente, ref) in enumerate(trabajos, 1):
    archivo = url_a_archivo.get(ref) if fuente == "web" else ref
    local = os.path.join(ruta, archivo or f"tmp_{n}.pdf")
    if archivo and archivo in pdf_drive:
        bajar_de_drive(pdf_drive[archivo], local)
    elif fuente == "web":
        contenido = descargar(ref)
        if contenido is None:
            control.append({"ARCHIVO": ref, "ESTADO": "ERROR DE DESCARGA"}); print(f"❌ [{n}] {ref[:90]}")
            continue
        open(local, "wb").write(contenido)
        time.sleep(PAUSA)
    r, fila = procesar(local, archivo or ref)
    tipo_w, anio_w, mes_w = enlaces.get(ref, (None, None, None)) if fuente == "web" else (None, None, None)
    if not r and fila["ESTADO"].startswith(("ESCANEADO", "FORMATO ANTIGUO")) and tipo_w and anio_w:
        # se guarda igual en Drive con nombre ordenado (servirá para la etapa de OCR)
        nombre = f"PP_{tipo_w}_{anio_w}" + (f"_{mes_w:02d}" if tipo_w == "MENSUAL" and mes_w else "") + ".pdf"
        url_a_archivo[ref] = nombre
        if chk_pdf.value and nombre not in pdf_drive:
            os.rename(local, os.path.join(ruta, nombre)); local = os.path.join(ruta, nombre)
            subir(local, PDF_FOLDER_ID, pdf_drive, mime="application/pdf")
        fila.update({"ARCHIVO": nombre, "TIPO": tipo_w, "AÑO": anio_w})
    if r:
        if not en_rango(r["TIPO"], r["ANIO"]):
            os.remove(local); continue
        nombre = f"PP_{r['TIPO']}_{r['ANIO']}" + (f"_{r['MES_REPORTE']:02d}" if r["TIPO"] == "MENSUAL" else "") + ".pdf"
        if fuente == "web":
            url_a_archivo[ref] = nombre
            if chk_pdf.value and nombre not in pdf_drive:
                os.rename(local, os.path.join(ruta, nombre)); local = os.path.join(ruta, nombre)
                subir(local, PDF_FOLDER_ID, pdf_drive, mime="application/pdf")
        fila["ARCHIVO"] = nombre
        resultados.append(r)
    control.append(fila)
    if os.path.exists(local):
        os.remove(local)
    print(f"{'✅' if r else '⚠️'} [{n}/{len(trabajos)}] {fila['ARCHIVO']}: "
          f"{len(r['filas']) if r else 0} filas · {fila.get('ESTADO', '')}")

pd.DataFrame([{"URL": u, "ARCHIVO": a} for u, a in url_a_archivo.items()], columns=["URL", "ARCHIVO"]).to_csv(
    os.path.join(ruta, NOMBRE_CATALOGO), index=False, encoding="utf-8-sig")
subir(os.path.join(ruta, NOMBRE_CATALOGO), DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 6. TABLA FINAL Y CONTROL ─────────────────────────────────────────────────
# Prioridad: el ANUAL de cada año; para años sin anual, la MENSUAL más reciente (trae el acumulado).
anuales = {r["ANIO"] for r in resultados if r["TIPO"] == "ANUAL"}
elegidos = {}
for r in resultados:
    if r["TIPO"] == "ANUAL":
        elegidos[(r["ANIO"], "A")] = r
    elif r["ANIO"] not in anuales:
        k = (r["ANIO"], "M")
        if k not in elegidos or r["MES_REPORTE"] > elegidos[k]["MES_REPORTE"]:
            elegidos[k] = r

filas, ctrl_final = [], []
for (anio, _), r in sorted(elegidos.items()):
    hasta = 12 if r["TIPO"] == "ANUAL" else r["MES_REPORTE"]
    archivo = f"PP_{r['TIPO']}_{anio}" + (f"_{r['MES_REPORTE']:02d}" if r["TIPO"] == "MENSUAL" else "") + ".pdf"
    df = pd.DataFrame(r["filas"])
    est = "OK" if len(df) else "SIN TABLA"
    fila_c = {"ARCHIVO": archivo, "TIPO": r["TIPO"], "AÑO": anio, "MESES": hasta, "LOTES": len(df),
              "FILAS_IMPUTADAS": int(df["ESTADO"].str.startswith("IMPUTADO").sum()) if len(df) else 0,
              "FILAS_REVISAR": int((df["ESTADO"] == "REVISAR").sum()) if len(df) else 0}
    # suma de lotes vs el SUB-TOTAL/TOTAL que imprime el PDF, mes a mes.
    # Anuales recientes: un subtotal por producto. Antiguos (p. ej. 2001): un solo TOTAL general.
    def dif_contra(sub, refs):
        difs = [max(abs(sub[m].sum() - t.get(m, 0)) for m in MESES[:hasta] if m in t) for t in refs]
        return min(difs) if difs else None
    dif_total = dif_contra(df, r["totales"]) if len(df) else None
    fila_c["DIF_MAX_TOTAL"] = dif_total
    # El TOTAL general de los anuales antiguos va al final de la tabla y queda marcado con el último
    # producto (LGN): no es un subtotal de ese producto y no se usa para su control.
    gran_total = float(df["TOTAL"].sum()) if len(df) else 0.0
    es_general = lambda t: df["PRODUCTO"].nunique() > 1 and abs((t.get("TOTAL") or 0) - gran_total) <= 1
    for prod in ("PETROLEO", "LGN", "CONDENSADO"):
        sub = df[df["PRODUCTO"] == prod] if len(df) else df
        refs = [t for t in r["totales"] if t["PRODUCTO"] == prod and not es_general(t)]
        dif = dif_contra(sub, refs) if len(sub) else 0.0
        fila_c[f"BARRILES_{prod}"] = round(float(sub["TOTAL"].sum()), 0) if len(sub) else 0
        fila_c[f"DIF_MAX_{prod}"] = dif
        if (dif is None or dif > 1) and not (dif_total is not None and dif_total <= 1):
            est = "REVISAR"
    if fila_c["FILAS_REVISAR"]:
        est = "REVISAR"
    fila_c["ESTADO"] = est
    ctrl_final.append(fila_c)
    for _, f in df.iterrows():
        for k, m in enumerate(MESES[:hasta], 1):
            bls = float(f[m] or 0)
            filas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": f["ZONA"],
                          **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], f["ZONA"]))),
                          "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                          "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                          "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": f["PRODUCTO"],
                          "BARRILES": bls, "BPD": round(bls / calendar.monthrange(anio, k)[1], 1),
                          "ESTADO_FILA": f["ESTADO"], "FUENTE": archivo})

# ── Años escaneados (1994–2000): transcripciones validadas guardadas en el repositorio ──
# Cada tabla se transcribió a mano desde la imagen y se validó: cada lote suma su total anual, cada mes suma
# el TOTAL del mes y el gran total coincide con el PDF.
URL_TRANSC = ("https://raw.githubusercontent.com/adriansivina-droid/scraper-mef/main/HIDROCARBUROS/"
              "transcripciones/PP_mensual_{}_transcrito.csv")
MES_TR = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SET", "OCT", "NOV", "DIC"]
anios_ok = {c["AÑO"] for c in ctrl_final if c["ESTADO"] == "OK"}
anios_transc = set()
for anio in range(A_INI, A_FIN + 1):
    if anio in anios_ok:
        continue
    try:
        rt = requests.get(URL_TRANSC.format(anio), timeout=30)
    except Exception:
        continue
    if rt.status_code != 200:
        continue
    t = pd.read_csv(io.StringIO(rt.text), dtype={"LOTE": str, "LOTE_ORIGINAL": str})
    tot, t = t[t["PRODUCTO"] == "TOTAL"], t[t["PRODUCTO"] != "TOTAL"]
    dif = max(abs(t[m].sum() - tot.iloc[0][m]) for m in MES_TR + ["TOTAL"]) if len(tot) else None
    filas_mal = int((t[MES_TR].sum(axis=1) != t["TOTAL"]).sum())
    fila_c = {"ARCHIVO": t["FUENTE"].iloc[0], "TIPO": "TRANSCRITO", "AÑO": anio, "MESES": 12, "LOTES": len(t),
              "FILAS_IMPUTADAS": 0, "FILAS_REVISAR": filas_mal, "DIF_MAX_TOTAL": dif,
              "ESTADO": "OK (transcrito)" if dif == 0 and filas_mal == 0 else "REVISAR"}
    for prod_ in ("PETROLEO", "LGN", "CONDENSADO"):
        fila_c[f"BARRILES_{prod_}"] = float(t.loc[t["PRODUCTO"] == prod_, "TOTAL"].sum())
    ctrl_final.append(fila_c)
    anios_transc.add(anio)
    for _, f in t.iterrows():
        for k, m in enumerate(MES_TR, 1):
            bls = float(f[m])
            filas.append({"AÑO": anio, "MES": k, "FECHA": f"01/{k:02d}/{anio}", "ZONA": zona_de(f["LOTE"]),
                          **dict(zip(("DEPARTAMENTO", "DEPARTAMENTO_FUENTE"), depto_de(f["LOTE"], zona_de(f["LOTE"])))),
                          "OPERADOR": norm_operador(f["OPERADOR"]), "OPERADOR_ORIGINAL": f["OPERADOR"],
                          "LOTE": f["LOTE"], "LOTE_HISTORICO": lote_historico(f["LOTE"]),
                          "LOTE_ORIGINAL": f["LOTE_ORIGINAL"], "PRODUCTO": f["PRODUCTO"],
                          "BARRILES": bls, "BPD": round(bls / calendar.monthrange(anio, k)[1], 1),
                          "ESTADO_FILA": "TRANSCRITO", "FUENTE": "TRANSCRIPCION " + f["FUENTE"]})
if anios_transc:
    print(f"✍️ Años tomados de transcripciones validadas: {', '.join(map(str, sorted(anios_transc)))}")

prod = pd.DataFrame(filas)
ctrl = pd.DataFrame(ctrl_final + [c for c in control if c.get("ESTADO", "").startswith(("ERROR", "NO ES", "ESCANEADO", "FORMATO"))
                                  and c.get("AÑO") not in anios_transc])
hoy = date.today().strftime("%d/%m/%Y")
ctrl["FECHA_EXTRACCION"] = hoy
NOMBRE_DATOS = f"PP_produccion_lote_mes_{A_INI}-{A_FIN}.csv"
NOMBRE_CONTROL = f"CONTROL PP produccion_lote_mes_{A_INI}-{A_FIN}.csv"
prod.to_csv(os.path.join(ruta, NOMBRE_DATOS), index=False, encoding="utf-8-sig")
ctrl.to_csv(os.path.join(ruta, NOMBRE_CONTROL), index=False, encoding="utf-8-sig")
subir(os.path.join(ruta, NOMBRE_DATOS), DRIVE_FOLDER_ID, datos_drive)
subir(os.path.join(ruta, NOMBRE_CONTROL), DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 7. RESUMEN ───────────────────────────────────────────────────────────────
print("═" * 70)
print(f"Años con datos: {prod['AÑO'].nunique() if len(prod) else 0} · filas: {len(prod):,}")
if len(ctrl):
    display(ctrl["ESTADO"].value_counts().to_frame("archivos"))
    rev = ctrl[~ctrl["ESTADO"].isin(["OK", "OK (transcrito)"])]
    if len(rev):
        antiguos = rev[rev["ESTADO"].str.startswith(("ESCANEADO", "FORMATO"), na=False)]
        if len(antiguos):
            print(f"🗂️ {len(antiguos)} PDF antiguos (escaneados): quedan guardados en DATA/PDF para la etapa de OCR · "
                  f"años: {', '.join(str(a) for a in sorted(antiguos['AÑO'].dropna().astype(int)))}")
        otros = rev.drop(antiguos.index)
        if len(otros):
            print("⚠️ Archivos a revisar (totales que no cuadran o error): mándalos para ajustar el extractor")
            display(otros)
if len(prod):
    lor = prod[(prod["DEPARTAMENTO"] == "LORETO") & (prod["PRODUCTO"] == "PETROLEO")]
    if len(lor):
        print("🛢️ Loreto · petróleo fiscalizado por lote histórico (millones de barriles; 1-AB y 192 = mismo campo):")
        tabla = (lor.pivot_table(index="LOTE_HISTORICO", columns="AÑO", values="BARRILES", aggfunc="sum") / 1e6).round(2)
        # 0 en los años en que el lote ya existía pero no figura en el PDF; vacío solo antes de su primer año
        tabla = tabla.where(tabla.notna() | ~tabla.notna().cummax(axis=1), 0.0)
        ult = int(lor["AÑO"].max()); meses_ult = int(lor.loc[lor["AÑO"] == ult, "MES"].max())
        if meses_ult < 12:
            tabla = tabla.rename(columns={ult: f"{ult} (ene–{MESES[meses_ult - 1][:3].lower()})"})
        tabla["ACUMULADO"] = tabla.sum(axis=1)
        display(tabla.sort_values("ACUMULADO", ascending=False))
print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")
